# One-Hot Encoding of EEG Labels

**Dataset**: MOABB BNCI2014-001 (Motor Imagery)  
**Channels**: 22 channels  
**Sampling rate**: 250 Hz  
**Subject**: 1

---

## Overview

We demonstrate one-hot encoding for multi-class EEG labels using all 4 motor imagery classes.

## What this notebook does

Applies LabelEncoder followed by OneHotEncoder to convert string labels into a binary matrix representation.

## What you should expect to see

- Bar chart showing the distribution of all 4 classes
- Heatmap of the one-hot encoded matrix for the first 20 trials
- Each trial has exactly one '1' in the column of its class

## Key parameters

| Parameter | Value |
| --- | --- |
| fmin | 8 |
| fmax | 32 |
| n_classes | 4 |
| encoder | LabelEncoder, OneHotEncoder |


## 1. Install dependencies


In [ ]:
!pip install moabb mne scipy numpy plotly scikit-learn


## 2. Load MOABB dataset

MOABB downloads data automatically on first use (~44 MB).


In [ ]:
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery
import numpy as np

dataset = BNCI2014_001()
paradigm = MotorImagery(n_classes=4, fmin=8, fmax=32)
X, labels, meta = paradigm.get_data(dataset=dataset, subjects=[1])

print(f'X shape: {X.shape}')
print(f'Labels: {np.unique(labels)}')
print(f'Trials: {len(labels)}')


In [ ]:
# Using all 4 classes (no filtering)
print(f'Classes: {np.unique(labels)}')
print(f'Class distribution: {[(c, np.sum(labels == c)) for c in np.unique(labels)]}')


## 3. Explore the data


In [ ]:
n_trials, n_channels, n_samples = X.shape
print(f'Trials: {n_trials}')
print(f'Channels: {n_channels}')
print(f'Samples per trial: {n_samples}')
print(f'Trial duration: {n_samples/250:.2f} s')


## 4. Apply one-hot encoding


In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

label_encoder = LabelEncoder()
labels_encoded = label_encoder.fit_transform(labels)

onehot_encoder = OneHotEncoder(sparse_output=False)
onehot_matrix = onehot_encoder.fit_transform(labels_encoded.reshape(-1, 1))

classes = label_encoder.classes_
print(f'Encoded labels: {np.unique(labels_encoded)}')
print(f'One-hot matrix shape: {onehot_matrix.shape}')
print(f'Classes: {classes}')


## 5. Interactive plot

**What to look for:**

- All 4 classes should have roughly equal counts (balanced dataset)
- The one-hot matrix shows a single '1' per row, confirming correct encoding
- Each column corresponds to one class


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

counts = [np.sum(labels == c) for c in classes]

fig = make_subplots(rows=2, cols=1, subplot_titles=(
    'Class Distribution (4 classes)',
    'One-Hot Encoded Matrix (first 20 trials)'))

fig.add_trace(go.Bar(x=list(classes), y=counts, marker_color='steelblue', name='Count'), row=1, col=1)
fig.add_trace(go.Heatmap(z=onehot_matrix[:20, :], x=list(classes),
    colorscale='Blues', name='One-Hot', showscale=True), row=2, col=1)

fig.update_xaxes(title_text='Class', row=1, col=1)
fig.update_yaxes(title_text='Count', row=1, col=1)
fig.update_xaxes(title_text='Class', row=2, col=1)
fig.update_yaxes(title_text='Trial', row=2, col=1)
fig.update_layout(height=800, showlegend=False, title_text='One-Hot Encoding of EEG Labels')
fig.show()


## What did we learn?

- One-hot encoding converts categorical labels into a binary matrix suitable for neural networks
- Each row has exactly one '1', representing the active class
- LabelEncoder maps strings to integers, OneHotEncoder expands integers to binary vectors
